# 1 - Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [3]:
from src.utils import config, io

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


# 2 - Preprocessing

In [4]:
X = io.load_csv(config.PROCESSED_DATA_DIR / 'X.csv', index_col=0)
y = io.load_csv(config.PROCESSED_DATA_DIR / 'y.csv', index_col=0)

In [6]:
split_cfg = io.load_json(config.PROCESSED_DATA_DIR / 'splits/temporal_v2.json')
split_cfg

{'description': 'Forecasting split for future country risk prediction',
 'train_years': [1999, 2021],
 'test_years': [2022, 2026]}

In [7]:
def get_subset_data(data, bounds):
    return data[(data['YEAR'] >= bounds[0]) & (data['YEAR'] <= bounds[1])]

In [8]:
X_train = get_subset_data(X, split_cfg['train_years'])
y_train = y.loc[X_train.index]
X_test = get_subset_data(X, split_cfg['test_years'])
y_test = y.loc[X_test.index]

# 3 - Train Model

In [13]:
from src.preprocessing import preprocess_pipeline
from src.models import model_pipeline, evaluate

In [ ]:
preprocessor_params = {
    'num_imputer': 'uni',
    'num_imputer_uni_strategy': 'mean',  # mean is correct for continuous macro indicators
    'cat_imputer': 'uni',
    'cat_imputer_uni_strategy': 'most_frequent'
}

In [15]:
preprocessor = preprocess_pipeline.build_preprocessor(X, preprocessor_params)

In [16]:
import mlflow

mlflow.set_tracking_uri(config.PROJECT_ROOT / 'models/mlruns')
mlflow.set_experiment('Country Risk Prediction')


<Experiment: artifact_location='file:///Users/hippolytegrandet/Desktop/Dev/country_risk_rating/models/mlruns/991689756472581023', creation_time=1770566631957, experiment_id='991689756472581023', last_update_time=1770566631957, lifecycle_stage='active', name='Country Risk Prediction', tags={}>

## 3.1 - Baseline, Logistic Regression Model

In [ ]:
model_name = 'logistic_regression'

model_params = { 
    'penalty': 'l2',
    'C' : 10,
    'solver': 'newton-cg',
    'max_iter'  : 1000
}

model = model_pipeline.get_model_pipeline(
    model_name,
    preprocessor,
    model_params
)

In [ ]:
with mlflow.start_run(run_name='baseline_lr_v1'):

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    test_metrics = evaluate.evaluate_model_classifier(model, X_test, y_test, prefix='test_')

    # Log split metadata
    mlflow.log_params({
        'model': model_name,
        **model_params,
        'train_years': split_cfg['train_years'],
        'test_years': split_cfg['test_years']
    })


    mlflow.log_metrics({**test_metrics})

    # Log model
    # mlflow.sklearn.log_model(
    #     model,
    #     artifact_path='model',
    #     registered_model_name=None,
    #     input_example=X_test.loc[[X_test.index[0]]]
    # )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

## 3.2 XGBoost Classifier

In [22]:
model_name = 'xgboost_classifier'

model_params = {
    'objective': 'multi:softprob', 
    'num_class': 7,

    'gamma': 0.0,	
    'max_depth': 10,
    'min_child_weight': 1.0,
    'colsample_bytree': 0.8,
    'subsample': 1.0
}

model = model_pipeline.get_model_pipeline(
    model_name,
    preprocessor,
    model_params
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

y_encoder = LabelEncoder()
y_encoder.fit(y_train.values.ravel())

In [27]:
type(model)

sklearn.pipeline.Pipeline

In [28]:
with mlflow.start_run(run_name='baseline_xgb_class_v1'):

    # Log split metadata
    mlflow.log_params({
        'model': model_name,
        **model_params,
        'train_years': split_cfg['train_years'],
        # 'val_years': split_cfg['val_years'],
        'test_years': split_cfg['test_years']
    })

    # Train
    model.fit(X_train, y_encoder.transform(y_train))

    # Evaluate
    test_metrics = evaluate.evaluate_model_classifier(model, X_test, y_encoder.transform(y_test), prefix='test_')

    mlflow.log_metrics(test_metrics)

    # Log model
    mlflow.sklearn.log_model(
        model,
        artifact_path='model',
        registered_model_name=None,
        input_example=X_test.loc[[X_test.index[0]]]
    )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
2026/04/10 21:09:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Classifier Results
test_accuracy: 0.8427
test_precision: 0.7937
test_recall: 0.7915
test_f1: 0.7867
test_blurred_accuracy: 0.9607
test_dist_accuracy_ratio: 0.2547

Classification Report
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       158
           1       0.75      0.79      0.77        19
           2       0.75      0.98      0.85        41
           3       0.81      0.57      0.67        30
           4       0.70      0.65      0.67        43
           5       0.68      0.77      0.72        69
           6       0.89      0.82      0.86       123

    accuracy                           0.84       483
   macro avg       0.79      0.79      0.79       483
weighted avg       0.85      0.84      0.84       483


Confusion Matrix
[[153   0   0   0   0   0   5]
 [  3  15   1   0   0   0   0]
 [  0   1  40   0   0   0   0]
 [  1   1   8  17   3   0   0]
 [  1   2   4   2  28   6   0]
 [  0   1   0   1   7  53   7]
 [  0   0   0

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in

MLflow run_id: ff0f1d6b928e41eca2de981a9dfc562d


In [29]:
xgb_cls_run_id = run_id
print('XGBoost classifier run_id:', xgb_cls_run_id)

XGBoost classifier run_id: ff0f1d6b928e41eca2de981a9dfc562d


## 3.3 - XGBoost Regressor

In [13]:
model_name = 'xgboost_regressor'

model_params = {
    'objective': 'reg:squarederror', 
    'gamma': 0.0,	
    'max_depth': 10,
    'min_child_weight': 1.0,
    'colsample_bytree': 0.6,
    'subsample': 1.0
}

model = model_pipeline.get_model_pipeline(
    model_name,
    preprocessor,
    model_params
)

In [30]:
with mlflow.start_run(run_name='baseline_xgb_reg_v1'):

    # Log split metadata
    mlflow.log_params({
        'model': model_name,
        **model_params,
        'train_years': split_cfg['train_years'],
        # 'val_years': split_cfg['val_years'],
        'test_years': split_cfg['test_years']
    })

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    test_metrics = evaluate.evaluate_model_regressor(model, X_test, y_test, prefix='test_')

    mlflow.log_metrics(test_metrics)

    # Log model
    # mlflow.xgboost.log_model(
    #     model,
    #     artifact_path='model',
    #     registered_model_name=None,
    #     input_example=X_test.loc[[X_test.index[0]]]
    # )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

Baseline Logistic Regression Results
test_accuracy: 0.5601
test_precision: 0.5154
test_recall: 0.5559
test_f1: 0.4872
test_blurred_accuracy: 0.8961
test_dist_accuracy_ratio: 0.5988
test_mae: 0.6837
test_mse: 1.0620

Classification Report
              precision    recall  f1-score   support

           1       1.00      0.62      0.76       159
           2       0.26      0.88      0.40        17
           3       0.43      0.49      0.46        43
           4       0.17      0.33      0.23        27
           5       0.35      0.38      0.36        48
           6       0.43      0.68      0.53        74
           7       0.97      0.52      0.68       123

    accuracy                           0.56       491
   macro avg       0.52      0.56      0.49       491
weighted avg       0.72      0.56      0.60       491


Confusion Matrix
[[98 32 13  9  6  1  0]
 [ 0 15  2  0  0  0  0]
 [ 0 10 21 12  0  0  0]
 [ 0  1  8  9  8  1  0]
 [ 0  0  3 15 18 12  0]
 [ 0  0  2  7 13 50  2]
 [ 

## 3.4 Ensemble XGBoost

In [33]:
reg_model_params = {
    'objective': 'reg:squarederror', 
    'gamma': 0.0,	
    'max_depth': 10,
    'min_child_weight': 1.0,
    'colsample_bytree': 0.6,
    'subsample': 1.0
}

model_reg = model_pipeline.get_model_pipeline(
    'xgboost_regressor',
    preprocessor,
    reg_model_params
)

In [ ]:
clas_model_params = {
    'objective': 'multi:softprob', 
    'num_class': 7,

    'gamma': 0.0,	
    'max_depth': 10,
    'min_child_weight': 1.0,
    'colsample_bytree': 0.8,
    'subsample': 1.0
}

model_clas = model_pipeline.get_model_pipeline(
    'xgboost_classifier',
    preprocessor,
    clas_model_params
)

from sklearn.preprocessing import LabelEncoder

y_encoder = LabelEncoder()
y_encoder.fit(y_train.values.ravel())

In [38]:
with mlflow.start_run(run_name='baseline_xgb_ensemble_v1'):

    # Log split metadata
    mlflow.log_params({
        'model': 'ensemble_clas_reg',
        **clas_model_params,
        **reg_model_params,
        'train_years': split_cfg['train_years'],
        # 'val_years': split_cfg['val_years'],
        'test_years': split_cfg['test_years']
    })

    # Train
    model_clas.fit(X_train, y_encoder.transform(y_train))
    model_reg.fit(X_train, y_train)
    models = {
        'clas': model_clas, 
        'reg': model_reg
    }

    # Evaluate
    test_metrics = evaluate.evaluate_ensemble_model(models, X_test, y_test, prefix='test_')

    mlflow.log_metrics(test_metrics)

    # Log model
    # mlflow.xgboost.log_model(
    #     model,
    #     artifact_path='model',
    #     registered_model_name=None,
    #     input_example=X_test.loc[[X_test.index[0]]]
    # )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


{'test_meanPred_accuracy': 0.6863543788187373, 'test_meanPred_precision': 0.6168507311664575, 'test_meanPred_recall': 0.6922826600912036, 'test_meanPred_f1': 0.6187104820375493, 'test_meanPred_blurred_accuracy': 0.9409368635437882, 'test_meanPred_dist_accuracy_ratio': np.float64(0.4134419551934827), 'test_meanPred_mae': 0.48371921258157236, 'test_meanPred_mse': 0.7108111288618851}
MLflow run_id: 187b013bf2d5414db878194e7bd904d0


In [94]:
import numpy as np

y_pred_clas = models['clas'].predict(X_test) + 1
y_pred_clas_proba = models['clas'].predict_proba(X_test)
y_pred_clas_weighted_sum = (y_pred_clas_proba * [i for i in range(1, 8)]).sum(axis=1)

y_pred_reg = models['reg'].predict(X_test)
y_pred_reg_class = np.round(np.clip(y_pred_reg, 1, 7))

y_pred_mean = (y_pred_clas + y_pred_reg) / 2
y_pred_mean_clas = np.round(np.clip(y_pred_mean, 1, 7))

y_pred_wighted_mean = (y_pred_clas_weighted_sum + y_pred_reg) / 2
y_pred_wighted_mean_clas = np.round(np.clip(y_pred_wighted_mean, 1, 7))

results = evaluate.evaluate_classification(y_test, y_pred_mean_clas, 'meanPred_')
results = results | evaluate.evaluate_classification(y_test, y_pred_wighted_mean_clas, 'WeightedMeanPred_')
results['meanPred_' + 'mae'] = evaluate.mean_absolute_error(y_test, y_pred_mean)
results['meanPred_' + 'mse'] = evaluate.mean_squared_error(y_test, y_pred_mean)


In [95]:
results

{'meanPred_accuracy': 0.6863543788187373,
 'meanPred_precision': 0.6168507311664575,
 'meanPred_recall': 0.6922826600912036,
 'meanPred_f1': 0.6187104820375493,
 'meanPred_blurred_accuracy': 0.9409368635437882,
 'meanPred_dist_accuracy_ratio': np.float64(0.4134419551934827),
 'WeightedMeanPred_accuracy': 0.6578411405295316,
 'WeightedMeanPred_precision': 0.5929172012034497,
 'WeightedMeanPred_recall': 0.6721682635962273,
 'WeightedMeanPred_f1': 0.590815502697502,
 'WeightedMeanPred_blurred_accuracy': 0.9164969450101833,
 'WeightedMeanPred_dist_accuracy_ratio': np.float64(0.4684317718940937),
 'meanPred_mae': 0.48371921258157236,
 'meanPred_mse': 0.7108111288618851}

# 4 - Register Best Model with MLflow

Registers the best-performing model (XGBoost classifier) into the MLflow Model Registry.  
Use `mlflow ui --backend-store-uri models/mlruns` to browse runs and manage model versions.

In [30]:
import mlflow

mlflow.set_tracking_uri(config.PROJECT_ROOT / 'models/mlruns')

model_uri = f"runs:/{xgb_cls_run_id}/model"
registered = mlflow.register_model(model_uri=model_uri, name="xgboost_classifier")

print(f"Registered: {registered.name}  version {registered.version}")

Registered model 'xgboost_classifier' already exists. Creating a new version of this model...
2026/04/10 21:09:48 WARNING mlflow.tracking._model_registry.fluent: Run with id ff0f1d6b928e41eca2de981a9dfc562d has no artifacts at artifact path 'model', registering model based on models:/m-7b75f6e1824f4d9e973c964bfe2c3b87 instead


Registered: xgboost_classifier  version 1


Created version '1' of model 'xgboost_classifier'.


In [ ]:
# Load a registered model back from the MLflow registry
# model = mlflow.sklearn.load_model(f"models:/xgboost_classifier/{registered.version}")